In [ ]:
import pandas as pd
import numpy as np
import os


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def remove_outliers_iqr(df, col, iqr_k=1.5):
    s = pd.to_numeric(df[col], errors="coerce")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - iqr_k * iqr
    high = q3 + iqr_k * iqr
    return df.loc[(s >= low) & (s <= high)].copy()

def cliffs_delta(x, y):
    x = pd.to_numeric(pd.Series(x).dropna(), errors="coerce").dropna().values
    y = pd.to_numeric(pd.Series(y).dropna(), errors="coerce").dropna().values
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan
    diffs = x[:, None] - y[None, :]
    n_greater = np.sum(diffs > 0)
    n_less = np.sum(diffs < 0)
    return float((n_greater - n_less) / (nx * ny))

def permutation_pvalue(x, y, n_perm=10000, seed=42):
    """Two-sided permutation test for mean difference"""
    rng = np.random.default_rng(seed)
    x = np.array(x.dropna())
    y = np.array(y.dropna())
    if len(x) == 0 or len(y) == 0:
        return np.nan
    obs_diff = np.abs(np.mean(x) - np.mean(y))
    combined = np.concatenate([x, y])
    nx = len(x)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(combined)
        x_perm, y_perm = combined[:nx], combined[nx:]
        diff = np.abs(np.mean(x_perm) - np.mean(y_perm))
        if diff >= obs_diff:
            count += 1
    return count / n_perm

def chunk_list(lst, n):
    """Split list into chunks of size n."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def plot_gap_by_diagnosis_clean_wrap(
    df,
    gap_col,
    diagnosis_list,
    thresholds=(0.1, 0.5, 1.0, 1.5),
    per_line=3,
    title_fontsize=8,
    n_perm=10000
):
    # Build original data by group
    group_data_orig = {}
    for diag in diagnosis_list:
        sub = df[df["clinical_diagnosis"] == diag][[gap_col]].copy()
        sub["Origen"] = diag
        group_data_orig[diag] = sub

    # Unique pairs without repetition
    all_pairs = [(diagnosis_list[i], diagnosis_list[j])
                 for i in range(len(diagnosis_list))
                 for j in range(i+1, len(diagnosis_list))]

    cn_pairs = [(a, b) for (a, b) in all_pairs if a == "CN"]
    other_pairs = [p for p in all_pairs if p not in cn_pairs]

    results_table = []

    fig, axes = plt.subplots(2, 2, figsize=(7.0, 8.0), sharey=True)
    axes = axes.flatten()

    for ax, iqr_k in zip(axes, thresholds):
        group_data_clean = {
            diag: remove_outliers_iqr(group_data_orig[diag], gap_col, iqr_k)
            for diag in diagnosis_list
        }

        cn_lines = []
        for a, b in cn_pairs:
            x = group_data_clean[a][gap_col]
            y = group_data_clean[b][gap_col]
            dval = cliffs_delta(x, y)
            pval = permutation_pvalue(x, y, n_perm=n_perm)
            cn_lines.append(f"{a}-{b}: {dval:.3f}")
            results_table.append([iqr_k, a, b, dval, pval])

        other_lines = []
        for a, b in other_pairs:
            x = group_data_clean[a][gap_col]
            y = group_data_clean[b][gap_col]
            dval = cliffs_delta(x, y)
            pval = permutation_pvalue(x, y, n_perm=n_perm)
            other_lines.append(f"{a}-{b}: {dval:.3f}")
            results_table.append([iqr_k, a, b, dval, pval])

        cn_wrapped = [" | ".join(chunk) for chunk in chunk_list(cn_lines, per_line)]
        other_wrapped = [" | ".join(chunk) for chunk in chunk_list(other_lines, per_line)]

        title_parts = [f"IQR={iqr_k}"] + cn_wrapped + other_wrapped
        ax.set_title("\n".join(title_parts), fontsize=title_fontsize)

        combined = pd.concat([group_data_clean[d] for d in diagnosis_list], ignore_index=True)
        sns.boxplot(data=combined, x="Origen", y=gap_col, order=diagnosis_list, ax=ax)
        ax.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_xlabel("")
        ax.set_ylabel(gap_col)

    plt.tight_layout()
    results_df = pd.DataFrame(results_table, columns=["IQR_k", "Group1", "Group2", "CliffsDelta", "p_value"])
    return fig, axes, results_df

# Modulation SAGs vs number of languages

In [ ]:
df_SAGs_cormob= pd.read_excel('../../Data/df_SAGs-covars.xlsx')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

df_plot = df_SAGs_cormob[['demo_language_num', 'GAP_corrected_M1']].copy()
df_plot = df_plot.dropna()

df_plot['demo_language_num'] = pd.to_numeric(df_plot['demo_language_num'], errors='coerce')
df_plot['GAP_corrected_M1'] = pd.to_numeric(df_plot['GAP_corrected_M1'], errors='coerce')
df_plot = df_plot.dropna()

x = df_plot['demo_language_num']
y = df_plot['GAP_corrected_M1']

slope, intercept, r_value, p_value, std_err = linregress(x, y)

plt.figure(figsize=(4, 3.5))

sns.regplot(
    data=df_plot,
    x='demo_language_num',
    y='GAP_corrected_M1',
    scatter_kws={'alpha': 0.5},
    line_kws={'color': 'red', 'linewidth': 2}
)

label_text = f"r = {r_value:.2f} | slope = {slope:.2f}"
plt.plot([], [], ' ', label=label_text)

plt.xlabel('Language number')
plt.ylabel('GAP corrected (M1)')
plt.title('GAP vs Language number')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()


plt.show()

# Ancovas

## Sex

### SAG ~ Diagnosis, sex, education country

In [ ]:
import statsmodels.api as sm

X1_sub = df_SAGs_cormob.copy()

X1_sub = X1_sub[['GAP_corrected_M1', 'clinical_diagnosis', 'demo_sex', 'cog_ed', 'demo_age', 'Country']]
X1_sub.dropna(inplace=True)
X1_sub.reset_index(drop=True, inplace=True)

formula = 'GAP_corrected_M1 ~ C(clinical_diagnosis) + C(demo_sex)  + cog_ed  + demo_age + C(Country) ' 

from statsmodels.formula.api import ols
model = ols(formula, data=X1_sub).fit()
X1_sub["GAP_residualized"] = model.resid

anova_table = sm.stats.anova_lm(model, typ=2)

anova_table

In [ ]:
formula = 'GAP_corrected_M1 ~ C(demo_sex)  + cog_ed  + demo_age + C(Country) ' 

from statsmodels.formula.api import ols
model = ols(formula, data=X1_sub).fit()
X1_sub["GAP_residualized"] = model.resid

diagnosis_list = ["CN", "DCL", "AD", "FTD", "FTD-L"]

a = plot_gap_by_diagnosis_clean_wrap(
    X1_sub,
    gap_col="GAP_residualized",
    diagnosis_list=diagnosis_list
);

In [ ]:
a[2][a[2].IQR_k == 1.5]

### Predictors ~ Diagnosis, sex, education country

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

predictors = [
    'concreteness', 'granularity_extraction', 'psycholinguistic_objective',
    'pitch_analysis', 'sentiment_analysis', 'talking_intervals', 'verbosity'
]

X1_sub = df_SAGs_cormob.copy()

X1_sub = X1_sub[['GAP_corrected_M1', 'clinical_diagnosis', 'demo_sex', 'cog_ed', 'demo_age', 'Country'] + predictors]

results = {}

for var in predictors:
    formula = f'{var} ~ C(demo_sex) + clinical_diagnosis + cog_ed + demo_age + C(Country)'
    model = ols(formula, data=X1_sub).fit()
    anova = sm.stats.anova_lm(model, typ=2)
    
    anova['variable'] = var
    results[var] = anova

#df_results = pd.concat(results)
results;

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

alpha = 0.05

df_plot = pd.concat(results).reset_index()
df_plot = df_plot.rename(columns={'level_0': 'predictor', 'level_1': 'term'})

resid = (
    df_plot[df_plot['term'] == 'Residual'][['predictor', 'sum_sq']]
    .rename(columns={'sum_sq': 'sum_sq_resid'})
)

df_plot = df_plot[df_plot['term'] != 'Residual'].copy()

df_plot = df_plot.merge(resid, on='predictor', how='left')

df_plot['eta2'] = df_plot['sum_sq'] / (df_plot['sum_sq'] + df_plot['sum_sq_resid'])

df_plot['sig'] = df_plot['PR(>F)'] < alpha

keep = ['C(demo_sex)', 'composite_cormob', 'demo_age', 'cog_ed', 'C(Country)']
df_plot = df_plot[df_plot['term'].isin(keep)].copy()

label_map = {
    'C(demo_sex)': 'Sex',
    'demo_age': 'Age',
    'cog_ed': 'Education',

}
df_plot['covariate'] = df_plot['term'].map(label_map)

order_y = [
    'concreteness',
    'granularity_extraction',
    'psycholinguistic_objective',
    'pitch_analysis',
    'sentiment_analysis',
    'talking_intervals',
    'verbosity'
]

df_plot['predictor'] = pd.Categorical(df_plot['predictor'], categories=order_y, ordered=True)
df_plot = df_plot.sort_values('predictor')

palette = {
    'Sex': 'black',
    'Age': 'black',
    'Education': 'black',
    'Country': 'black'
}

markers = {
    'Sex': 'o',
    'Age': '^',
    'Education': 'D',
    'Country': 'P'
}

fig, ax = plt.subplots(figsize=(9, 3))

ax.axvspan(0, 0.01, color='lightgray', alpha=0.15)
ax.axvspan(0.01, 0.06, color='khaki', alpha=0.18)
ax.axvspan(0.06, 0.14, color='orange', alpha=0.12)
ax.axvspan(0.14, max(df_plot['eta2'].max() * 1.05, 0.15), color='tomato', alpha=0.10)

ax.axvline(0.01, linestyle='--', color='gray', linewidth=1)
ax.axvline(0.06, linestyle='--', color='gray', linewidth=1)
ax.axvline(0.14, linestyle='--', color='gray', linewidth=1)

for cov in ['Sex', 'Age', 'Education', 'Country']:
    sub = df_plot[df_plot['covariate'] == cov]
    
    for _, row in sub.iterrows():
        ax.scatter(
            row['eta2'],
            row['predictor'],
            s=60 + row['eta2'] * 2500,
            color=palette[cov],
            marker=markers[cov],
            alpha=0.95 if row['sig'] else 0.18,
            edgecolor='black' if row['sig'] else 'none',
            linewidth=0.6
        )

ax.set_xlabel('Effect size (partial η²)')
ax.set_ylabel('Linguistic predictors')
ax.set_title('Influence of covariates on linguistic predictors')

legend_elements = [
    Line2D([0], [0], marker=markers['Sex'], color='w', label='Sex',
           markerfacecolor=palette['Sex'], markeredgecolor='black', markersize=8),

    Line2D([0], [0], marker=markers['Age'], color='w', label='Age',
           markerfacecolor=palette['Age'], markeredgecolor='black', markersize=8),
    Line2D([0], [0], marker=markers['Education'], color='w', label='Education',
           markerfacecolor=palette['Education'], markeredgecolor='black', markersize=8),

    Line2D([0], [0], color='lightgray', lw=8, alpha=0.5, label='Small (<0.01)'),
    Line2D([0], [0], color='khaki', lw=8, alpha=0.7, label='Medium (0.01–0.06)'),
    Line2D([0], [0], color='orange', lw=8, alpha=0.6, label='Large (0.06–0.14)'),
    Line2D([0], [0], color='tomato', lw=8, alpha=0.5, label='Very large (>0.14)')
]

ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

## Sex and cormobilities (composite score)

### SAG ~ Diagnosis, sex, education country

In [ ]:

X1_sub = df_SAGs_cormob.copy()


X1_sub = X1_sub[['GAP_corrected_M1', 'clinical_diagnosis', 'demo_sex', 'cog_ed', 'demo_age', 'Country', 'composite_cormob']]
X1_sub.dropna(inplace=True)
X1_sub.reset_index(drop=True, inplace=True)

formula = 'GAP_corrected_M1 ~ C(clinical_diagnosis) + C(demo_sex)  + cog_ed  + demo_age + C(Country) + composite_cormob'

from statsmodels.formula.api import ols
model = ols(formula, data=X1_sub).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

#anova_table.to_excel('anova_HC_LACvsHC_nonLAC.xlsx')
anova_table

In [ ]:
formula = 'GAP_corrected_M1 ~ + C(demo_sex)  + cog_ed  + demo_age + C(Country) + composite_cormob' 

from statsmodels.formula.api import ols
model = ols(formula, data=X1_sub).fit()
X1_sub["GAP_residualized"] = model.resid

diagnosis_list = ["CN", "DCL", "AD", "FTD", "FTD-L"]

a = plot_gap_by_diagnosis_clean_wrap(
    X1_sub,
    gap_col="GAP_residualized",
    diagnosis_list=diagnosis_list
);

In [ ]:
a[2][a[2].IQR_k == 1.5]

### Predictors ~ Diagnosis, sex, education country

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

predictors = [
    'concreteness', 'granularity_extraction', 'psycholinguistic_objective',
    'pitch_analysis', 'sentiment_analysis', 'talking_intervals', 'verbosity'
]

X1_sub = df_SAGs_cormob.copy()

X1_sub = X1_sub[['GAP_corrected_M1', 'clinical_diagnosis', 'demo_sex', 'cog_ed', 'demo_age', 'Country', 'composite_cormob'] + predictors] 

results = {}

for var in predictors:
    formula = f'{var} ~ C(demo_sex) + C(clinical_diagnosis) + cog_ed + demo_age + C(Country) + composite_cormob'
    model = ols(formula, data=X1_sub).fit()
    anova = sm.stats.anova_lm(model, typ=2)
    
    anova['variable'] = var
    results[var] = anova

results;

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

alpha = 0.05

df_plot = pd.concat(results).reset_index()
df_plot = df_plot.rename(columns={'level_0': 'predictor', 'level_1': 'term'})


resid = (
    df_plot[df_plot['term'] == 'Residual'][['predictor', 'sum_sq']]
    .rename(columns={'sum_sq': 'sum_sq_resid'})
)

df_plot = df_plot[df_plot['term'] != 'Residual'].copy()

df_plot = df_plot.merge(resid, on='predictor', how='left')

df_plot['eta2'] = df_plot['sum_sq'] / (
    df_plot['sum_sq'] + df_plot['sum_sq_resid']
)

df_plot['sig'] = df_plot['PR(>F)'] < alpha

keep = [
    'C(demo_sex)',
    'composite_cormob',
    'demo_age',
    'cog_ed',
]

df_plot = df_plot[df_plot['term'].isin(keep)].copy()

label_map = {
    'C(demo_sex)': 'Sex',
    'composite_cormob': 'Composite comorbidity',
    'demo_age': 'Age',
    'cog_ed': 'Education',
    'C(Country)': 'Country'
}

df_plot['covariate'] = df_plot['term'].map(label_map)

order_y = [
    'concreteness',
    'granularity_extraction',
    'psycholinguistic_objective',
    'pitch_analysis',
    'sentiment_analysis',
    'talking_intervals',
    'verbosity'
]

df_plot['predictor'] = pd.Categorical(
    df_plot['predictor'],
    categories=order_y,
    ordered=True
)

df_plot = df_plot.sort_values('predictor')

palette = {
    'Sex': 'black',
    'Composite comorbidity': 'black',
    'Age': 'black',
    'Education': 'black',

}

markers = {
    'Sex': 'o',
    'Composite comorbidity': 's',
    'Age': '^',
    'Education': 'D',
    'Country': 'P'
}

fig, ax = plt.subplots(figsize=(9, 3))

ax.axvspan(0, 0.01, color='lightgray', alpha=0.15)
ax.axvspan(0.01, 0.06, color='khaki', alpha=0.18)
ax.axvspan(0.06, 0.14, color='orange', alpha=0.12)
ax.axvspan(
    0.14,
    max(df_plot['eta2'].max() * 1.05, 0.15),
    color='tomato',
    alpha=0.10
)

ax.axvline(0.01, linestyle='--', color='gray', linewidth=1)
ax.axvline(0.06, linestyle='--', color='gray', linewidth=1)
ax.axvline(0.14, linestyle='--', color='gray', linewidth=1)

# Scatter
for cov in ['Sex', 'Composite comorbidity', 'Age', 'Education', 'Country']:
    sub = df_plot[df_plot['covariate'] == cov]

    for _, row in sub.iterrows():
        ax.scatter(
            row['eta2'],
            row['predictor'],
            s=60 + row['eta2'] * 2500,
            color=palette[cov],
            marker=markers[cov],
            alpha=0.95 if row['sig'] else 0.18,
            edgecolor='black' if row['sig'] else 'none',
            linewidth=0.6
        )

ax.set_xlabel('Effect size (partial η²)')
ax.set_ylabel('Linguistic predictors')
ax.set_title('Influence of covariates on linguistic predictors')

legend_elements = [
    Line2D([0], [0], marker=markers['Sex'], color='w', label='Sex',
           markerfacecolor=palette['Sex'], markeredgecolor='black', markersize=8),

    Line2D([0], [0], marker=markers['Composite comorbidity'], color='w',
           label='Composite comorbidity',
           markerfacecolor=palette['Composite comorbidity'],
           markeredgecolor='black', markersize=8),

    Line2D([0], [0], marker=markers['Age'], color='w', label='Age',
           markerfacecolor=palette['Age'], markeredgecolor='black', markersize=8),

    Line2D([0], [0], marker=markers['Education'], color='w', label='Education',
           markerfacecolor=palette['Education'], markeredgecolor='black', markersize=8),



    Line2D([0], [0], color='lightgray', lw=8, alpha=0.5, label='Small (<0.01)'),
    Line2D([0], [0], color='khaki', lw=8, alpha=0.7, label='Medium (0.01–0.06)'),
    Line2D([0], [0], color='orange', lw=8, alpha=0.6, label='Large (0.06–0.14)'),
    Line2D([0], [0], color='tomato', lw=8, alpha=0.5, label='Very large (>0.14)')
]

ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

## Sex and cormobilities (composite score) + number of languages

### SAG ~ Diagnosis, sex, education country

In [ ]:

X1_sub = df_SAGs_cormob.copy()


X1_sub = X1_sub[['GAP_corrected_M1', 'clinical_diagnosis', 'demo_sex', 'cog_ed', 'demo_age', 'Country', 'demo_language_num', 'composite_cormob']]
X1_sub.dropna(inplace=True)
X1_sub.reset_index(drop=True, inplace=True)

formula = 'GAP_corrected_M1 ~ C(clinical_diagnosis) + C(demo_sex)  + cog_ed  + demo_age + C(Country) + demo_language_num + composite_cormob'

from statsmodels.formula.api import ols
model = ols(formula, data=X1_sub).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

#anova_table.to_excel('anova_HC_LACvsHC_nonLAC.xlsx')
anova_table

In [ ]:
formula = 'GAP_corrected_M1 ~ C(demo_sex)  + cog_ed  + demo_age + C(Country) + demo_language_num + composite_cormob'

from statsmodels.formula.api import ols
model = ols(formula, data=X1_sub).fit()
X1_sub["GAP_residualized"] = model.resid

diagnosis_list = ["CN", "DCL", "AD", "FTD", "FTD-L"]

a = plot_gap_by_diagnosis_clean_wrap(
    X1_sub,
    gap_col="GAP_residualized",
    diagnosis_list=diagnosis_list
);

In [ ]:
a[2][a[2].IQR_k == 1.5]

### Predictors ~ Diagnosis, sex, education country

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

predictors = [
    'concreteness', 'granularity_extraction', 'psycholinguistic_objective',
    'pitch_analysis', 'sentiment_analysis', 'talking_intervals', 'verbosity'
]

X1_sub = df_SAGs_cormob.copy()

X1_sub = X1_sub[['GAP_corrected_M1', 'clinical_diagnosis', 'demo_sex', 'cog_ed', 'demo_age', 'Country', 'demo_language_num','composite_cormob'] + predictors] 

results = {}

for var in predictors:
    formula = f'{var} ~ C(demo_sex) + C(clinical_diagnosis) + cog_ed + demo_age + C(Country) + demo_language_num + composite_cormob'
    model = ols(formula, data=X1_sub).fit()
    anova = sm.stats.anova_lm(model, typ=2)
    
    anova['variable'] = var
    results[var] = anova

#df_results = pd.concat(results)
results;

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

alpha = 0.05

df_plot = pd.concat(results).reset_index()
df_plot = df_plot.rename(columns={'level_0': 'predictor', 'level_1': 'term'})

resid = (
    df_plot[df_plot['term'] == 'Residual'][['predictor', 'sum_sq']]
    .rename(columns={'sum_sq': 'sum_sq_resid'})
)

df_plot = df_plot[df_plot['term'] != 'Residual'].copy()

df_plot = df_plot.merge(resid, on='predictor', how='left')

df_plot['eta2'] = df_plot['sum_sq'] / (df_plot['sum_sq'] + df_plot['sum_sq_resid'])

df_plot['sig'] = df_plot['PR(>F)'] < alpha

keep = [
    'C(demo_sex)',
    'composite_cormob',
    'demo_age',
    'cog_ed',

    'C(Language)'   
]

df_plot = df_plot[df_plot['term'].isin(keep)].copy()

label_map = {
    'C(demo_sex)': 'Sex',
    'composite_cormob': 'Composite comorbidity',
    'demo_age': 'Age',
    'cog_ed': 'Education',
    'C(Country)': 'Country',
    'C(Language)': 'Language'   
}

df_plot['covariate'] = df_plot['term'].map(label_map)

order_y = [
    'concreteness',
    'granularity_extraction',
    'psycholinguistic_objective',
    'pitch_analysis',
    'sentiment_analysis',
    'talking_intervals',
    'verbosity'
]

df_plot['predictor'] = pd.Categorical(df_plot['predictor'], categories=order_y, ordered=True)
df_plot = df_plot.sort_values('predictor')

palette = {
    'Sex': 'black',
    'Composite comorbidity': 'black',
    'Age': 'black',
    'Education': 'black',
    'Country': 'black',
    'Language': 'black'   
}

markers = {
    'Sex': 'o',
    'Composite comorbidity': 's',
    'Age': '^',
    'Education': 'D',
    'Country': 'P',
    'Language': 'X'  
}

fig, ax = plt.subplots(figsize=(9, 3))

ax.axvspan(0, 0.01, color='lightgray', alpha=0.15)
ax.axvspan(0.01, 0.06, color='khaki', alpha=0.18)
ax.axvspan(0.06, 0.14, color='orange', alpha=0.12)
ax.axvspan(0.14, max(df_plot['eta2'].max() * 1.05, 0.15), color='tomato', alpha=0.10)

ax.axvline(0.01, linestyle='--', color='gray', linewidth=1)
ax.axvline(0.06, linestyle='--', color='gray', linewidth=1)
ax.axvline(0.14, linestyle='--', color='gray', linewidth=1)

# Scatter
for cov in ['Sex', 'Composite comorbidity', 'Age', 'Education', 'Country', 'Language']:
    sub = df_plot[df_plot['covariate'] == cov]

    for _, row in sub.iterrows():
        ax.scatter(
            row['eta2'],
            row['predictor'],
            s=60 + row['eta2'] * 2500,
            color=palette[cov],
            marker=markers[cov],
            alpha=0.95 if row['sig'] else 0.18,
            edgecolor='black' if row['sig'] else 'none',
            linewidth=0.6
        )

ax.set_xlabel('Effect size (partial η²)')
ax.set_ylabel('Linguistic predictors')
ax.set_title('Influence of covariates on linguistic predictors')

# Leyenda
legend_elements = [
    Line2D([0], [0], marker=markers['Sex'], color='w', label='Sex',
           markerfacecolor=palette['Sex'], markeredgecolor='black', markersize=8),
    Line2D([0], [0], marker=markers['Composite comorbidity'], color='w', label='Composite comorbidity',
           markerfacecolor=palette['Composite comorbidity'], markeredgecolor='black', markersize=8),
    Line2D([0], [0], marker=markers['Age'], color='w', label='Age',
           markerfacecolor=palette['Age'], markeredgecolor='black', markersize=8),
    Line2D([0], [0], marker=markers['Education'], color='w', label='Education',
           markerfacecolor=palette['Education'], markeredgecolor='black', markersize=8),

    Line2D([0], [0], marker=markers['Language'], color='w', label='Language',
           markerfacecolor=palette['Language'], markeredgecolor='black', markersize=8),
    Line2D([0], [0], color='lightgray', lw=8, alpha=0.5, label='Small (<0.01)'),
    Line2D([0], [0], color='khaki', lw=8, alpha=0.7, label='Medium (0.01–0.06)'),
    Line2D([0], [0], color='orange', lw=8, alpha=0.6, label='Large (0.06–0.14)'),
    Line2D([0], [0], color='tomato', lw=8, alpha=0.5, label='Very large (>0.14)')
]

ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

# Data preparation for R-matching

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def remove_outliers_iqr(df, col, iqr_k=1.5):
    s = pd.to_numeric(df[col], errors="coerce")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - iqr_k * iqr
    high = q3 + iqr_k * iqr
    return df.loc[(s >= low) & (s <= high)].copy()

def cliffs_delta(x, y):
    x = pd.to_numeric(pd.Series(x).dropna(), errors="coerce").dropna().values
    y = pd.to_numeric(pd.Series(y).dropna(), errors="coerce").dropna().values
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan
    diffs = x[:, None] - y[None, :]
    n_greater = np.sum(diffs > 0)
    n_less = np.sum(diffs < 0)
    return float((n_greater - n_less) / (nx * ny))

def permutation_pvalue(x, y, n_perm=10000, seed=42):
    """Two-sided permutation test for mean difference"""
    rng = np.random.default_rng(seed)
    x = np.array(x.dropna())
    y = np.array(y.dropna())
    if len(x) == 0 or len(y) == 0:
        return np.nan
    obs_diff = np.abs(np.mean(x) - np.mean(y))
    combined = np.concatenate([x, y])
    nx = len(x)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(combined)
        x_perm, y_perm = combined[:nx], combined[nx:]
        diff = np.abs(np.mean(x_perm) - np.mean(y_perm))
        if diff >= obs_diff:
            count += 1
    return count / n_perm

def chunk_list(lst, n):
    """Split list into chunks of size n."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def plot_gap_by_diagnosis_clean_wrap(
    df,
    gap_col,
    diagnosis_list,
    thresholds=(0.1, 0.5, 1.0, 1.5),
    per_line=3,
    title_fontsize=8,
    n_perm=10000
):
    # Build original data by group
    group_data_orig = {}
    for diag in diagnosis_list:
        sub = df[df["clinical_diagnosis"] == diag][[gap_col]].copy()
        sub["Origen"] = diag
        group_data_orig[diag] = sub

    # Unique pairs without repetition
    all_pairs = [(diagnosis_list[i], diagnosis_list[j])
                 for i in range(len(diagnosis_list))
                 for j in range(i+1, len(diagnosis_list))]

    cn_pairs = [(a, b) for (a, b) in all_pairs if a == "CN"]
    other_pairs = [p for p in all_pairs if p not in cn_pairs]

    results_table = []

    fig, axes = plt.subplots(2, 2, figsize=(7.0, 8.0), sharey=True)
    axes = axes.flatten()

    for ax, iqr_k in zip(axes, thresholds):
        group_data_clean = {
            diag: remove_outliers_iqr(group_data_orig[diag], gap_col, iqr_k)
            for diag in diagnosis_list
        }

        cn_lines = []
        for a, b in cn_pairs:
            x = group_data_clean[a][gap_col]
            y = group_data_clean[b][gap_col]
            dval = cliffs_delta(x, y)
            pval = permutation_pvalue(x, y, n_perm=n_perm)
            cn_lines.append(f"{a}-{b}: {dval:.3f}")
            results_table.append([iqr_k, a, b, dval, pval])

        other_lines = []
        for a, b in other_pairs:
            x = group_data_clean[a][gap_col]
            y = group_data_clean[b][gap_col]
            dval = cliffs_delta(x, y)
            pval = permutation_pvalue(x, y, n_perm=n_perm)
            other_lines.append(f"{a}-{b}: {dval:.3f}")
            results_table.append([iqr_k, a, b, dval, pval])

        cn_wrapped = [" | ".join(chunk) for chunk in chunk_list(cn_lines, per_line)]
        other_wrapped = [" | ".join(chunk) for chunk in chunk_list(other_lines, per_line)]

        title_parts = [f"IQR={iqr_k}"] + cn_wrapped + other_wrapped
        ax.set_title("\n".join(title_parts), fontsize=title_fontsize)

        combined = pd.concat([group_data_clean[d] for d in diagnosis_list], ignore_index=True)
        sns.boxplot(data=combined, x="Origen", y=gap_col, order=diagnosis_list, ax=ax)
        ax.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_xlabel("")
        ax.set_ylabel(gap_col)

    plt.tight_layout()
    results_df = pd.DataFrame(results_table, columns=["IQR_k", "Group1", "Group2", "CliffsDelta", "p_value"])
    return fig, axes, results_df

In [ ]:
df_all = pd.read_excel('../../Data/SAG-sex-matched.xlsx')

In [ ]:
diagnosis_list = ["CN", "DCL", "AD", "FTD", "FTD-L"]

a = plot_gap_by_diagnosis_clean_wrap(
    df_all,
    gap_col="GAP_corrected_M1",
    diagnosis_list=diagnosis_list
);


In [ ]:
a[2][a[2].IQR_k == 1.5]

# Network comparison

## Functional

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, linregress

networks = ['lgbm_dmn', 'lgbm_en', 'lgbm_mn', 'lgbm_sn', 'lgbm_vn']

name_map = {
    'lgbm_dmn': 'DMN',
    'lgbm_en': 'EN',
    'lgbm_mn': 'MN',
    'lgbm_sn': 'SN',
    'lgbm_vn': 'VN'
}

color_map = {
    'lgbm_dmn': '#1f77b4',
    'lgbm_en': '#ff7f0e',
    'lgbm_mn': '#2ca02c',
    'lgbm_sn': '#d62728',
    'lgbm_vn': '#9467bd'
}

plt.figure(figsize=(4, 3.5))

for i in networks:
    df_merge = pd.read_excel('../../Data/SAG-no-linguistic-' + i +'-.xlsx')


    df_plot = df_merge[['GAP_corrected_M1', 'BAG_corrected']].dropna().copy()

    if len(df_plot) < 3:
        continue

    x = df_plot['GAP_corrected_M1']
    y = df_plot['BAG_corrected']

    r, p = pearsonr(x, y)
    d = (2 * r) / np.sqrt(1 - r**2) if abs(r) < 1 else np.nan

    label = f"{name_map[i]} (r={r:.2f}, d={d:.2f})"

    plt.scatter(
        x, y,
        s=10,
        alpha=0.5,
        color=color_map[i],
        label=label
    )

    slope, intercept, _, _, _ = linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = intercept + slope * x_line

    plt.plot(
        x_line, y_line,
        color=color_map[i],
        linewidth=2
    )

plt.xlabel('GAP_corrected_M1', fontsize=12)
plt.ylabel('BAG', fontsize=12)

plt.legend(title='Network')

plt.tight_layout()
plt.xlim([-30, 30])
plt.ylim([-30, 30])

plt.show()

## Languague Network BAGs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, linregress

networks = [ 'whole-Brain', 'fedorenko-language-network', 'dual-stream-model']

name_map = {
    'fedorenko-language-network': 'FLN',
    'dual-stream-model': 'DSM',
    'whole-Brain':'whole-Brain [wB]'
}

color_map = {
    'fedorenko-language-network': '#1f77b4',  
    'dual-stream-model': '#d62728',          
    'whole-Brain': '#2ca02c',          
}

plt.figure(figsize=(4, 3.5))

for i in networks:
    
    df_merge = pd.read_excel('../../Data/SAG-linguistic-' + i +'-.xlsx')

    df_plot = df_merge[['GAP_corrected_M1', 'BAG_corrected']].dropna().copy()

    if len(df_plot) < 3:
        continue

    x = df_plot['GAP_corrected_M1']
    y = df_plot['BAG_corrected']

    r, p = pearsonr(x, y)
    d = (2 * r) / np.sqrt(1 - r**2) if abs(r) < 1 else np.nan

    label = f"{name_map[i]} (r={r:.2f}, d={d:.2f})"

    plt.scatter(
        x, y,
        s=10,
        alpha=0.4,
        color=color_map[i],
        label=label
    )

    slope, intercept, _, _, _ = linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = intercept + slope * x_line

    plt.plot(
        x_line, y_line,
        color=color_map[i],
        linewidth=2
    )

plt.xlabel('GAP_corrected_M1', fontsize=12)
plt.ylabel('BAG', fontsize=12)

plt.legend(title='Network')

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)


plt.xlim([-40, 40])
plt.ylim([-40, 40])
plt.xlabel('SAGs')

plt.tight_layout()


plt.show()